# Pix2Pix — corrida base y ablación sin GAN (Colab)

1. Monta Drive, clona el repo e instala dependencias.
2. Extrae `datasetv1.zip` en `data/processed/`.
3. Entrena con `configs/base.yaml` (salida en Drive: `Redes-Pix2Pix/resultados_pix2pix/base`).
4. Entrena la ablación `use_gan: false`, `lambda_l1: 1000` (salida en `.../resultados_pix2pix/ablation`).
5. Dibuja curvas y descarga un zip ligero (sin checkpoints; esos quedan en Drive).
6. Resume la última época de cada corrida.

La primera ejecución se detuvo en la celda 1 porque el zip no estaba en la ruta esperada; la celda ya busca el zip en las rutas usadas por el equipo.


In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/Redes-Pix2Pix')
REPO = Path('/content/Trabajo-Final-de-Redes')

# El zip del dataset estuvo en dos lugares distintos según quién corrió el notebook.
# La primera ejecución falló aquí (AssertionError): ahora se prueba cada ruta en orden.
CANDIDATOS = [
    DRIVE / 'datasetv1.zip',
    Path('/content/drive/MyDrive/datasetv1.zip'),
    Path('/content/datasetv1.zip'),
    Path('/content/datasetv1(1).zip'),
]
ZIP = next((p for p in CANDIDATOS if p.is_file()), None)
assert ZIP is not None, 'Sube datasetv1.zip a una de estas rutas: ' + ', '.join(map(str, CANDIDATOS))
print('Dataset:', ZIP)

if not (REPO / '.git').is_dir():
    subprocess.run(
        [
            'git',
            'clone',
            'https://github.com/Isabela1105/Trabajo-Final-de-Redes.git',
            str(REPO),
        ],
        check=True,
    )

os.chdir(REPO)
# Si el repo ya estaba clonado de una sesión anterior, traer la última versión de main.
subprocess.run(['git', 'pull', '-q', '--ff-only'], check=False)

# Colab ya trae TensorFlow con GPU; instalamos el resto de dependencias sin versiones rígidas
reqs = []
for line in Path('requirements.txt').read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    pkg = re.split(r'[<>=!~]', line)[0].strip()
    if pkg.lower().startswith('tensorflow'):
        continue
    reqs.append(pkg)

if reqs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *reqs], check=True)

import tensorflow as tf
from src.gan.config import load_config

print('Repo listo en:', os.getcwd())
print('TensorFlow:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
import zipfile
from pathlib import Path
from src.gan.config import load_config
from src.gan.data.tf_dataset import list_pairs

cfg = load_config('configs/base.yaml', fixed_seed=False)
DATA = Path(cfg.data['train_dir']).parent  # <repo>/data/processed

with zipfile.ZipFile(ZIP) as z:
    names = [n for n in z.namelist() if n.strip() and not n.startswith('__MACOSX')]
    tops = {n.split('/')[0] for n in names}
    if 'data' in tops:
        dest = REPO
    elif 'processed' in tops:
        dest = REPO / 'data'
    elif {'train', 'val', 'test'} & tops:
        dest = DATA
    else:
        dest = DATA
    z.extractall(dest)

print('Dataset extraído en:', dest)

for split in ('train', 'val', 'test'):
    d = DATA / split
    inputs, targets = list_pairs(d)
    n_in = len(list((d / 'input').glob('*.png')))
    n_tg = len(list((d / 'target').glob('*.png')))
    print(f'{split}: {len(inputs)} pares (input={n_in}, target={n_tg})')

In [ ]:
import subprocess
import sys
from pathlib import Path
from src.gan.config import load_config

# En Drive (no en /content): si Colab se desconecta, los checkpoints sobreviven y RESUME funciona.
OUT_ROOT = DRIVE / 'resultados_pix2pix'
BASE_OUT = OUT_ROOT / 'base'
BASE_OUT.mkdir(parents=True, exist_ok=True)

link = REPO / 'outputs' / 'base'
link.parent.mkdir(exist_ok=True)
if link.is_symlink() or link.exists():
    if link.is_symlink():
        link.unlink()
    else:
        raise RuntimeError(f'{link} existe y no es un symlink; revisa antes de continuar')
link.symlink_to(BASE_OUT, target_is_directory=True)

cfg = load_config('configs/base.yaml', fixed_seed=False)
assert cfg.output_dir.resolve() == BASE_OUT.resolve(), (cfg.output_dir, BASE_OUT)
print('checkpoints:', cfg.checkpoint_dir)
print('metrics.csv:', cfg.metrics_path)
print('logs       :', cfg.log_dir)
print('samples    :', cfg.samples_dir)

RESUME = False  # Cambia a True si se interrumpe y quieres continuar
cmd = [sys.executable, '-m', 'src.gan.train', '--config', 'configs/base.yaml']
if RESUME:
    cmd.append('--resume')

subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import sys
import yaml
from pathlib import Path

# En Drive (no en /content): si Colab se desconecta, los checkpoints sobreviven y RESUME funciona.
OUT_ROOT = DRIVE / 'resultados_pix2pix'
ABL_OUT = OUT_ROOT / 'ablation'
ABL_OUT.mkdir(parents=True, exist_ok=True)

with open('configs/base.yaml', encoding='utf-8') as f:
    cfg_dict = yaml.safe_load(f)

with open('configs/ablation_l1.yaml', encoding='utf-8') as f:
    abl = yaml.safe_load(f)

# PR #28: sin GAN y peso L1 mayor; el resto queda igual que la corrida principal
cfg_dict['train']['use_gan'] = abl['train']['use_gan']        # False
cfg_dict['train']['lambda_l1'] = abl['train']['lambda_l1']    # 1000.0
cfg_dict['output']['dir'] = str(ABL_OUT)

out_cfg = REPO / 'configs' / 'ablation_colab.yaml'
with out_cfg.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg_dict, f, sort_keys=False, allow_unicode=True)

print('Config de ablación:', out_cfg)

RESUME_ABL = False  # Cambia a True para reanudar la ablación
cmd = [sys.executable, '-m', 'src.gan.train', '--config', str(out_cfg)]
if RESUME_ABL:
    cmd.append('--resume')

subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path
import shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display
from google.colab import files

# En Drive (no en /content): si Colab se desconecta, los checkpoints sobreviven y RESUME funciona.
OUT_ROOT = DRIVE / 'resultados_pix2pix'
METRICS = OUT_ROOT / 'base' / 'metrics.csv'
OUT_PNG = OUT_ROOT / 'curvas_entrenamiento.png'

assert METRICS.is_file(), f'No existe {METRICS}'

df = pd.read_csv(METRICS)
df['epoch'] = pd.to_numeric(df['epoch'], errors='coerce')
df = df.dropna(subset=['epoch'])

columns = [
    ('loss_g', 'Loss G'),
    ('loss_d', 'Loss D'),
    ('l1', 'L1'),
    ('psnr', 'PSNR'),
    ('ssim', 'SSIM'),
]
columns = [(c, t) for c, t in columns if c in df.columns]

fig, axes = plt.subplots(len(columns), 1, figsize=(9, 3.2 * len(columns)), squeeze=False)
for ax, (col, title) in zip(axes.ravel(), columns):
    for split in ('train', 'val'):
        sub = (
            df[df['split'] == split]
            .groupby('epoch', as_index=False)[col]
            .mean()
            .sort_values('epoch')
        )
        if not sub.empty:
            ax.plot(sub['epoch'], sub[col], marker='o', linewidth=1.6, label=split)
    ax.set_title(title)
    ax.set_xlabel('época')
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()
fig.savefig(OUT_PNG, dpi=140, bbox_inches='tight')
plt.close(fig)

print('Gráfica guardada en:', OUT_PNG)
display(Image(filename=str(OUT_PNG)))

# Empaqueta métricas, configs, muestras y curvas. Los checkpoints (cientos de MB) se quedan
# en Drive: descargarlos en un zip llenaba el disco de Colab y no hacen falta en el PC.
PAQUETE = Path('/content/resultados_pix2pix_ligero')
if PAQUETE.exists():
    shutil.rmtree(PAQUETE)
shutil.copytree(OUT_ROOT, PAQUETE, ignore=shutil.ignore_patterns('checkpoints', 'ckpt-*'))
zip_path = shutil.make_archive('/content/resultados_pix2pix', 'zip', PAQUETE)
print('Archivo listo para descargar:', zip_path)
files.download(zip_path)

In [ ]:
from pathlib import Path
import pandas as pd

# En Drive (no en /content): si Colab se desconecta, los checkpoints sobreviven y RESUME funciona.
OUT_ROOT = DRIVE / 'resultados_pix2pix'

for run_name, label in [('base', 'Pix2Pix Completo (GAN + L1)'), ('ablation', 'Ablación (Solo L1, sin GAN)')]:
    csv_path = OUT_ROOT / run_name / 'metrics.csv'
    if not csv_path.is_file():
        print(f'=== {label}: todavía no hay {csv_path} (corre su celda de entrenamiento) ===')
        continue
    df = pd.read_csv(csv_path)
    df['epoch'] = pd.to_numeric(df['epoch'], errors='coerce')
    df = df.dropna(subset=['epoch'])

    # Filtramos estrictamente entrenamiento y validación
    df_tv = df[df['split'].isin(['train', 'val'])]
    last_epoch = df_tv['epoch'].max()
    resumen = (
        df_tv[df_tv['epoch'] == last_epoch]
        .groupby('split', as_index=False)
        .mean(numeric_only=True)
    )
    print(f'=== {label} (Última época: {int(last_epoch)}) ===')
    print(resumen.to_string(index=False))
    print()